# LightOnOCR

In [ ]:
%pip install -q boto3 huggingface_hub pypdfium2 Pillow

In [42]:
import os
import requests

url = os.environ.get(
    "PREDICTOR_URL",
    "http://lightonocr-2-1b-predictor.canberra.svc.cluster.local:8080/v1/chat/completions",
     
)
# "model" must match InferenceService metadata.name (vLLM --served-model-name={{.Name}})
# This model needs <image>\n at the *start* of the user text, then the instruction.
EXAMPLE_URL ="https://www.mattmahoney.net/ocr/numbers_gs150.jpg"
#EXAMPLE_URL = "https://huggingface.co/datasets/hf-internal-testing/fixtures_ocr/resolve/main/SROIE-receipt.jpeg"

user_prompt = os.environ.get("LIGHTON_OCR_PROMPT", "<image>\nExtract the items and total price from this receipt.")
payload = {
    "model": os.environ.get("INFERENCE_MODEL", "lightonocr-2-1b"),
    "messages": [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": user_prompt},
                {
                    "type": "image_url",
                    "image_url": {"url": os.environ.get("LIGHTON_OCR_IMAGE_URL", EXAMPLE_URL)},
                },
            ],
        }
    ],
    "max_tokens": int(os.environ.get("MAX_TOKENS", "1024")),
    "temperature": 0.0,
}
h = {"Content-Type": "application/json"}
tok = os.environ.get("INFERENCE_BEARER_TOKEN") or os.environ.get("OC_TOKEN")
if tok:
    h["Authorization"] = f"Bearer {tok}"
r = requests.post(url, json=payload, headers=h, timeout=300, verify=False)
if r.status_code == 200:
    print("✅ OCR Result:")
    print(r.json()["choices"][0]["message"]["content"])
else:
    print(f"❌ Error: {r.status_code}")
    print(r.text[:2000])

✅ OCR Result:
Extract the items and total price from this receipt.

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th></th>
      <th></th>
      <th></th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>3.75 miles</td>
      <td>7.5</td>
      <td>11.25</td>
      <td>15.0</td>
    </tr>
    <tr>
      <td>30.0</td>
      <td>33.75</td>
      <td>37.5</td>
      <td>41.25</td>
    </tr>
    <tr>
      <td>38</td>
      <td>117</td>
      <td>155</td>
      <td>240</td>
    </tr>
    <tr>
      <td>696</td>
      <td>757</td>
      <td>859</td>
      <td>959</td>
    </tr>
    <tr>
      <td>38</td>
      <td>129</td>
      <td>215</td>
      <td>318</td>
    </tr>
    <tr>
      <td>38</td>
      <td>120</td>
      <td>209</td>
      <td>300</td>
    </tr>
    <tr>
      <td>37.49</td>
      <td>117.50</td>
      <td>202.22</td>
      <td>257.02</td>
    </tr>
    <tr>
      <td>7:19:27</td>
      <td>8:34:52</td>
      <td>

In [43]:
%pip install -q boto3 huggingface_hub pypdfium2 Pillow
import base64
import requests
import pypdfium2 as pdfium
import io

# Ensure this matches your predictor URL
ENDPOINT = "http://lightonocr-2-1b-predictor.canberra.svc.cluster.local:8080/v1/chat/completions"
MODEL = "lightonocr-2-1b"

# 1. Download PDF
pdf_url = "https://arxiv.org/pdf/2412.13663"
pdf_data = requests.get(pdf_url).content

# 2. Convert PDF page to image
pdf = pdfium.PdfDocument(pdf_data)
page = pdf[0]
pil_image = page.render(scale=2.0).to_pil() # Lowered scale slightly for memory stability

# 3. Convert to base64
buffer = io.BytesIO()
pil_image.save(buffer, format="JPEG", quality=85) # JPEG is usually better for OCR token limits
image_base64 = base64.b64encode(buffer.getvalue()).decode('utf-8')

# 4. Make request with explicit TEXT PROMPT
payload = {
    "model": MODEL,
    "messages": [{
        "role": "user",
        "content": [
            {
                "type": "text", 
                "text": "<image>\nTranscribe the text from this page accurately."
            },
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
            }
        ]
    }],
    "max_tokens": 1024, # 4096 is very high for T4 memory, start lower
    "temperature": 0.2,
    "top_p": 0.9,
    "repetition_penalty": 1.2 # Added to prevent the !!!! loops
}

response = requests.post(ENDPOINT, json=payload)

if response.status_code == 200:
    text = response.json()['choices'][0]['message']['content']
    print(text)
else:
    print(f"Error {response.status_code}: {response.text}")

Note: you may need to restart the kernel to use updated packages.
arXiv:2412.13663v2 [cs.CL] 19 Dec 2024

# Smarter, Better, Faster, Longer: A Modern Bidirectional Encoder for Fast, Memory Efficient, and Long Context Finetuning and Inference

Benjamin Warner $^{1\dagger}$ Antoine Chaffin $^{2\dagger}$ Benjamin Clavié $^{1\dagger}$  
Orion Weller $^{3}$ Oskar Hallström $^{2}$ Said Taghadouini $^{2}$  
Alexis Gallagher $^{1}$ Raja Biswas $^{1}$ Faisal Ladhak $^{4*}$ Tom Aarsen $^{5}$  
Nathan Cooper $^{1}$ Griffin Adams $^{1}$ Jeremy Howard $^{1}$ Iacopo Poli $^{2}$

$^{1}$ Answer.AI $^{2}$ LightOn $^{3}$ Johns Hopkins University $^{4}$ NVIDIA $^{5}$ HuggingFace  
$\dagger$ : core authors, $*$ : work done while at Answer.AI

Correspondence: {bw,bc}@answer.ai, antoine.chaffin@lighton.ai

## Abstract

Encoder-only transformer models such as BERT offer a great performance-size tradeoff for retrieval and classification tasks with respect to larger decoder-only models. Despite being the workh